## Gold Layer — Taxpayer Compliance Summary

Purpose: transform the four cleaned Silver tables (one row per invoice,
declaration, or violation) into a single row per taxpayer that answers the
core business question — "who needs compliance attention?"

Approach: aggregate each Silver table separately first (invoices, 
declarations, violations), using a CTE (WITH clause) for each. Joining the
raw detail tables directly on taxpayer_id would multiply rows (a "fan-out"):
a taxpayer with 3 invoices and 4 violations would produce 3 × 4 = 12 rows
instead of 1. Pre-aggregating each source to one row per taxpayer before
joining avoids this.

`silver_taxpayers` is the base table (not one of the transaction tables),
so every taxpayer appears exactly once even if they have zero invoices,
declarations, or violations — LEFT JOIN + COALESCE(..., 0) handles this.
### Risk Score Methodology

`risk_score` is a weighted sum of compliance issues:
- Anomalous invoices × 2
- Late declarations × 2
- Missing declarations × 3
- Violations × 3

Weights reflect a judgment call: not filing at all, or an official recorded
violation, is treated as more serious than being late or having one
questionable invoice. These weights are adjustable and would ideally be
validated against real ZATCA penalty severity in a production setting —
here they represent a reasonable starting assumption for demonstration.


In [1]:
%%sql
CREATE OR REPLACE TABLE gold_taxpayer_compliance_summary AS

WITH invoice_summary AS (
    SELECT taxpayer_id,
           COUNT(*) AS total_invoices,
           SUM(vat_anomaly_flag) AS anomalous_invoices
    FROM silver_invoices
    GROUP BY taxpayer_id
),
declaration_summary AS (
    SELECT taxpayer_id,
           COUNT(*) AS total_declarations,
           SUM(CASE WHEN status = 'Late' THEN 1 ELSE 0 END) AS late_declarations,
           SUM(CASE WHEN status = 'Not Submitted' THEN 1 ELSE 0 END) AS missing_declarations,
           ROUND(SUM(outstanding_amount), 2) AS total_outstanding
    FROM silver_declarations
    GROUP BY taxpayer_id
),
violation_summary AS (
    SELECT taxpayer_id,
           COUNT(*) AS total_violations,
           ROUND(SUM(penalty_amount_sar), 2) AS total_penalties
    FROM silver_violations
    GROUP BY taxpayer_id
)
SELECT
    t.taxpayer_id,
    t.company_name,
    t.sector,
    t.city,
    COALESCE(i.total_invoices, 0) AS total_invoices,
    COALESCE(i.anomalous_invoices, 0) AS anomalous_invoices,
    COALESCE(d.total_declarations, 0) AS total_declarations,
    COALESCE(d.late_declarations, 0) AS late_declarations,
    COALESCE(d.missing_declarations, 0) AS missing_declarations,
    COALESCE(d.total_outstanding, 0) AS total_outstanding,
    COALESCE(v.total_violations, 0) AS total_violations,
    COALESCE(v.total_penalties, 0) AS total_penalties,
     (COALESCE(i.anomalous_invoices, 0) * 2)
        + (COALESCE(d.late_declarations, 0) * 2)
        + (COALESCE(d.missing_declarations, 0) * 3)
        + (COALESCE(v.total_violations, 0) * 3) AS risk_score,
    CASE WHEN total_violations > 3 THEN 1 ELSE 0 END AS red_flag
    
FROM silver_taxpayers t
LEFT JOIN invoice_summary i ON t.taxpayer_id = i.taxpayer_id
LEFT JOIN declaration_summary d ON t.taxpayer_id = d.taxpayer_id
LEFT JOIN violation_summary v ON t.taxpayer_id = v.taxpayer_id


StatementMeta(, cb733aaf-ff21-45cc-a8d7-1a16f0e2f145, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
%%sql
select * from gold_taxpayer_compliance_summary

StatementMeta(, cb733aaf-ff21-45cc-a8d7-1a16f0e2f145, 3, Finished, Available, Finished, False)

<Spark SQL result set with 800 rows and 14 fields>

In [13]:
%%sql
SELECT * FROM gold_taxpayer_compliance_summary ORDER BY risk_score DESc

StatementMeta(, 70143051-13d3-4415-ae87-2c7a297d753a, 14, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 10 fields>

In [1]:
%%sql
SELECT COUNT(*) FROM gold_taxpayer_compliance_summary WHERE risk_score >= 10

StatementMeta(, ada85920-ee86-4b39-99a2-1cb09f86b509, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [1]:
%%sql
SELECT red_flag, COUNT(*) FROM gold_taxpayer_compliance_summary GROUP BY red_flag

StatementMeta(, d9ca353c-1439-4722-aa77-74ac4db1a144, 2, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

## Gold Layer — Sector Summary

Rolls the taxpayer-level Gold table up one level, from per-taxpayer to
per-sector, to answer "which industry sector shows the highest compliance
risk?" Built directly from `gold_taxpayer_compliance_summary` rather than
re-joining the Silver tables — since that table is already one row per
taxpayer, a simple GROUP BY sector with SUM/AVG is sufficient; no
fan-out risk here.

In [22]:
%%sql
CREATE OR REPLACE TABLE gold_sector_summary AS
SELECT
    sector,
    COUNT(*) AS total_taxpayers,
    SUM(total_invoices) AS total_invoices,
    SUM(anomalous_invoices) AS total_anomalous_invoices,
    SUM(total_declarations) AS total_declarations,
    SUM(late_declarations) AS total_late_declarations,
    SUM(missing_declarations) AS total_missing_declarations,
    ROUND(SUM(total_outstanding), 2) AS total_outstanding_vat,
    SUM(total_violations) AS total_violations,
    ROUND(SUM(total_penalties), 2) AS total_penalties,
    ROUND(AVG(risk_score), 2) AS avg_risk_score
FROM gold_taxpayer_compliance_summary
GROUP BY sector

StatementMeta(, 70143051-13d3-4415-ae87-2c7a297d753a, 23, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [24]:
%%sql
SELECT * FROM gold_sector_summary ORDER BY avg_risk_score DESC

StatementMeta(, 70143051-13d3-4415-ae87-2c7a297d753a, 25, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 11 fields>

In [25]:
%%sql
SELECT sector, avg_risk_score
FROM gold_sector_summary
ORDER BY avg_risk_score DESC

StatementMeta(, 70143051-13d3-4415-ae87-2c7a297d753a, 26, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 2 fields>

In [1]:
%%sql
CREATE OR REPLACE TABLE dim_date AS
SELECT
    date_add('2023-01-01', pos) AS date_key,
    YEAR(date_add('2023-01-01', pos)) AS year,
    MONTH(date_add('2023-01-01', pos)) AS month_number,
    DATE_FORMAT(date_add('2023-01-01', pos), 'MMMM') AS month_name,
    DATE_FORMAT(date_add('2023-01-01', pos), 'yyyy-MM') AS year_month,
    QUARTER(date_add('2023-01-01', pos)) AS quarter
FROM (
    SELECT EXPLODE(SEQUENCE(0, 1460)) AS pos
)

StatementMeta(, 509e9a17-5b3c-4510-b5e7-bb2a16747773, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
%%sql
SELECT * FROM dim_date LIMIT 10

StatementMeta(, 509e9a17-5b3c-4510-b5e7-bb2a16747773, 3, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 6 fields>